In [1]:
import random
import torch
from tensordict import TensorDict
from routerl import TrafficEnvironment

In [2]:
human_learning_episodes = 100

env_params = {
    "agent_parameters" : {
        "num_agents" : 100,
        "new_machines_after_mutation": 20,
        "human_parameters" : {
            "model" : "gawron"
        },
        "machine_parameters" : {
            "behavior" : "selfish",
        }
    },
    "simulator_parameters" : {
        "network_name" : "two_route_yield",
        # "network_name" : "cologne",
        # "sumo_type": "sumo-gui"
    },  
    "plotter_parameters" : {
        "phases" : [0, human_learning_episodes], # the number of episodes human learning will take
    },
}

In [3]:
env = TrafficEnvironment(seed=48, **env_params)

env.start()

for episode in range(human_learning_episodes):
    env.step()

env.mutation()

[CONFIRMED] Environment variable exists: SUMO_HOME
[SUCCESS] Added module directory: /usr/share/sumo/tools
 Retrying in 1 seconds


In [4]:
# Q-Table
q_table = {str(machine.id): [0, 0] for machine in env.machine_agents}

In [5]:
# Training
lr = 0.35
exploration_eps = 0.7
n_epochs = 1000

for epoch in range(n_epochs):
    cur_exploration_eps = exploration_eps * (1 - epoch / n_epochs)

    agent_action_list = {}

    env.reset()
    for agent in env.agent_iter():
        observation, reward, termination, truncation, info = env.last()

        if termination or truncation:
            last_action = agent_action_list[agent]
            q_table[agent][last_action] = lr * reward + (1 - lr) * q_table[agent][last_action]

            action = None
        else:
            if random.random() < cur_exploration_eps:
                action = env.action_space(agent).sample()
            else:
                action = 0 if q_table[agent][0] > q_table[agent][1] else 1
            
            agent_action_list[agent] = action
        
        env.step(action)

In [6]:
env.plot_results()

12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] findfont: Font family 'Times New Roman' not found.
12:47:45 [WARNING] f